# GWAS-PPI 解析パイプライン (pj02)

## 概要
疾患はゲノム構造の影響を受けた下流の遺伝子が引き起こす生体メカニズムの異常と定義する。
上流（ゲノム）から下流（疾患メカニズム）の間で中心を担う遺伝子を見つけることで創薬標的探索を見出す。

### パイプライン
1. **GWAS SNP → 遺伝子マッピング**: 特定疾患のSNPsを調査し、SNP関連遺伝子を特定
2. **PPI ネットワーク構築**: SNP関連遺伝子と繋がりのある遺伝子群を取得 (1～3階層)
3. **フローベース中心性解析**: GWAS P値 + variant影響度をフローとし、重要遺伝子をスコア化
4. **エンリッチメント解析**: Reactome, GO, HPO によるローカル解析
5. **可視化**: サンキーダイアグラム、ネットワーク図、パスウェイ-遺伝子マトリックス

---
## 0. セットアップ

In [1]:
import sys
import os

# プロジェクトルートをパスに追加
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import config
import gwas_fetcher
import ppi_fetcher
import gene_scorer
import network_analysis
import enrichment
import visualization
import id_mapper

import pandas as pd
import numpy as np
import networkx as nx
import plotly.io as pio

pio.renderers.default = 'notebook'

print('✅ 全モジュール読み込み完了')
print(f'出力ディレクトリ: {config.OUTPUT_DIR}')
import target_fetcher


✅ 全モジュール読み込み完了
出力ディレクトリ: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output


---
## 【付録】疾患の検索キー (ID) の探し方
下のセルに調べたい疾患名（英語）を入力して実行すると、GWAS Catalogが推奨する最新のID（例: `MONDO_...` や `EFO_...`）を探すことができます。
見つけたIDをコピーして、次の「1. パラメータ設定」セルの `TARGET_DISEASE_ID` に貼り付けてください。

In [2]:
import id_resolver

# ここに英語の疾患キーワードを入れて実行
id_resolver.search_disease_id("ulcerative colitis")

'ulcerative colitis' に関連するGWAS Catalog上の疾患IDを検索中...

検索APIエラー: ステータスコード 404


---
## 1. パラメータ設定
ここで解析対象の疾患名やパラメータを設定します。

In [3]:
# ========================================
# ★ ここを変更して解析対象を設定 ★
# ========================================

# 疾患の検索キー (GWAS Catalog 検索用 / 最も推奨される検索キー)
# ※ GWAS Catalog では新しいオントロジー (MONDO等) が採用されています
TARGET_DISEASE_ID = "MONDO_0005148"  # Type 2 Diabetes

# --- 以下の情報は id_resolver によって自動的に補完されます ---
import id_resolver
disease_info = id_resolver.auto_resolve_disease(TARGET_DISEASE_ID)
DISEASE_TRAIT = disease_info.get('trait') or 'Type 2 Diabetes'
MESH_ID = disease_info.get('mesh_id') or 'D003924'
EFO_TRAIT_ID = TARGET_DISEASE_ID

# PPI ソース選択 (デフォルト: SIGNOR, Reactome, STRING(Score>0.5))
PPI_SOURCES = ["signor", "reactome", "string"]

# PPI 階層数 (1～3, デフォルト: 1)
PPI_LAYERS = 1

# STRING スコア閾値 (500 = 0.5)
STRING_MIN_SCORE = 500

# エンリッチメントに使用するデータベース
ENRICHMENT_DBS = ["reactome", "go", "hpo"]

# FDR 閾値
FDR_THRESHOLD = 0.05

# 重要遺伝子の最大数
TOP_N_KEY_GENES = 50

print(f'\n【解析パラメータ】')
print(f'疾患: {DISEASE_TRAIT}')
print(f'対象ID: {TARGET_DISEASE_ID}')
print(f'MeSH ID: {MESH_ID}')
print(f'PPI ソース: {PPI_SOURCES}')
print(f'PPI 階層: {PPI_LAYERS}')
print(f'STRING閾値: {STRING_MIN_SCORE/1000:.1f}')
print(f'Enrichment DB: {ENRICHMENT_DBS}')


[Resolver] IDを解決中: MONDO_0005148 ...
  [GWAS Catalog] Trait名を取得: type 2 diabetes mellitus
  [OLS] MeSH IDを特定: D003924

【解析パラメータ】
疾患: type 2 diabetes mellitus
対象ID: MONDO_0005148
MeSH ID: D003924
PPI ソース: ['signor', 'reactome', 'string']
PPI 階層: 1
STRING閾値: 0.7
Enrichment DB: ['reactome', 'go', 'hpo']


---
## 2. GWAS SNP → 遺伝子マッピング
特定の疾患のSNPsを調査し、SNPに関連のある遺伝子（SNP関連遺伝子）を特定する。

In [4]:
# GWAS データ取得
gwas_df = pd.DataFrame()

# EFO_TRAIT_IDで一括取得すると数万件に及びAPIがハングアップ(タイムアウト)するため、
# id_resolverで取得した正確な疾患名(DISEASE_TRAIT)を使って、スタディ単位で安全に分割取得します。
# (configの DEFAULT_MAX_GWAS_STUDIES 制限が効くようになります)
if DISEASE_TRAIT:
    print(f'\n[GWAS Fetch] 疾患名 "{DISEASE_TRAIT}" でスタディ単位の取得を開始します...')
    gwas_df = gwas_fetcher.fetch_snps_for_disease(DISEASE_TRAIT)
else:
    print('DISEASE_TRAIT が設定されていません。')

print(f'\n取得結果:')
print(f'  SNPs: {gwas_df["rsid"].nunique()}')
print(f'  遺伝子: {gwas_df["gene_symbol"].nunique()}')
print(f'  アソシエーション: {len(gwas_df)}')

if not gwas_df.empty:
    gwas_genes = gwas_df['gene_symbol'].unique().tolist()
    print(f'\nSNP関連遺伝子リスト ({len(gwas_genes)}): {gwas_genes[:20]}...')
else:
    print('\n⚠️ アソシエーションが取得できませんでした。検索IDまたは疾患名を見直してください。')

gwas_df.head(10)



[GWAS Fetch] 疾患名 "type 2 diabetes mellitus" でスタディ単位の取得を開始します...

GWAS Catalog: 'type 2 diabetes mellitus' の SNPs を検索中 (最大 50 スタディ)...
[GWAS] 'type 2 diabetes mellitus' で 50 スタディを取得
  [1/50] Study GCST000073 を処理中...
    → 0 アソシエーション (閾値未満)
  [2/50] Study GCST000074 を処理中...
    → 0 アソシエーション (閾値未満)
  [3/50] Study GCST000075 を処理中...
    → 0 アソシエーション (閾値未満)
  [4/50] Study GCST000076 を処理中...
    → 0 アソシエーション (閾値未満)
  [5/50] Study GCST000049 を処理中...
    → 1 アソシエーション
  [6/50] Study GCST000277 を処理中...
    → 3 アソシエーション
  [7/50] Study GCST000024 を処理中...
    → 9 アソシエーション
  [8/50] Study GCST000025 を処理中...
    → 6 アソシエーション
  [9/50] Study GCST000012 を処理中...
    → 1 アソシエーション
  [10/50] Study GCST000167 を処理中...
    → 11 アソシエーション
  [11/50] Study GCST000219 を処理中...
    → 1 アソシエーション
  [12/50] Study GCST000221 を処理中...
    → 3 アソシエーション
  [13/50] Study GCST000383 を処理中...
    → 7 アソシエーション
  [14/50] Study GCST000665 を処理中...
    → 2 アソシエーション
  [15/50] Study GCST000712 を処理中...
    → 25 アソシエーション
  [16/50] Study G

,rsid,gene_symbol,p_value,study_id
0,rs2237896,KCNQ1,3.000000e-70,GCST003400
1,rs7901695,TCF7L2,1.000000e-48,GCST000025
2,rs2237892,KCNQ1,2.000000e-42,GCST000219
3,rs35612982,CDKAL1,6.000000e-36,GCST003400
4,rs2383208,CDKN2B,2.000000e-29,GCST000383
5,rs2383208,CDKN2A,2.000000e-29,GCST000383
6,rs7756992,CDKAL1,2.000000e-26,GCST002352
7,rs1470579,IGFBP2,2.000000e-24,GCST003400
8,rs34872471,TCF7L2,3.000000e-23,GCST003400
9,rs1552224,CENTD2,1.000000e-22,GCST000712


---
## 2.5 創薬ターゲットの取得
PharosおよびChEMBLから、疾患に関連する既知のターゲット情報を取得します。

In [5]:
known_targets = target_fetcher.get_combined_targets(
    disease_name=DISEASE_TRAIT,
    mesh_id=MESH_ID,
    efo_id=EFO_TRAIT_ID
)
target_fetcher.save_targets(known_targets)

print(f'取得したターゲット数: {len(known_targets)}')
if known_targets:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(list(known_targets.items()), columns=['Gene', 'Max Phase']).head(10))


[Target] Pharos APIを検索中: type 2 diabetes mellitus
[Target] Pharos APIから 6 件のターゲットを取得しました。
[Target] ChEMBL APIを検索中: type 2 diabetes mellitus (MeSH: D003924, EFO: MONDO_0005148)
[Target] ChEMBL APIから 191 件のターゲットを取得しました。
[Target] 【統合結果】合計 194 件の固有ターゲットを抽出しました。
[Target] 既知ターゲット情報を保存しました: data/known_targets.csv
取得したターゲット数: 194


,Gene,Max Phase
0,GCK,1
1,IRS1,1
2,INSR,4
3,PPARG,4
4,KCNJ11,4
5,ENPP1,1
6,MGAM,4
7,PDE5A,2
8,CA2,3
9,ESR1,1


---
## 3. PPI ネットワーク構築
SNP関連遺伝子と繋がりのある遺伝子群（PPI遺伝子群）を調べる。
- Sources: SIGNOR, Reactome, STRING (Score>0.5)
- 階層: 設定値 (デフォルト: 1)

In [ ]:
# 多階層 PPI 取得
ppi_df = ppi_fetcher.fetch_multi_layer_ppi(
    seed_genes=gwas_genes,
    layers=PPI_LAYERS,
    sources=PPI_SOURCES,
    string_min_score=STRING_MIN_SCORE,
)

if not ppi_df.empty:
    ppi_genes = set(ppi_df['gene_a']) | set(ppi_df['gene_b'])
    ppi_only_genes = ppi_genes - set(gwas_genes)
    print(f'\nPPI遺伝子 (GWAS外): {len(ppi_only_genes)}')
    display(ppi_df.head(10))


---
## 4. 遺伝子スコアリング
SNP関連遺伝子のフロー（初期流量）を定義:
- GWAS P値 (-log10)
- LoF, GoF, Missense 等の variant 影響度
- PPI マルチソーススコア

In [ ]:
# バリアント機能アノテーション取得 (Ensembl VEP)
rsids = gwas_df['rsid'].unique().tolist()
print(f'{len(rsids)} バリアントのアノテーション取得中...')

variant_consequences = gene_scorer.fetch_variant_consequences(rsids)
variant_classes = gene_scorer.classify_variant_effects(variant_consequences)

if not variant_classes.empty:
    print(f'\nバリアント分類結果:')
    print(f'  LoF: {variant_classes["is_lof"].sum()}')
    print(f'  GoF: {variant_classes["is_gof"].sum()}')
    print(f'  Missense: {variant_classes["is_missense"].sum()}')


In [ ]:
# 遺伝子スコアリング (GWAS P値 + VEP + PPIマルチソース)
gene_scores_df = gene_scorer.score_genes(
    gwas_df=gwas_df,
    variant_classifications=variant_classes,
    ppi_df=ppi_df,
)

print(f'\n遺伝子スコア (Top 15):')
gene_scores_df.head(15)


---
## 5. ネットワーク構築 & RWR 解析
PPIネットワークを構築し、バリアントスコアを初期流量としてRWR（Random Walk with Restart）を実行。
バリアントスコアの高い遺伝子からのシグナルがネットワーク上を伝播し、重要な中継遺伝子が浮かび上がる。

In [ ]:
import importlib
importlib.reload(network_analysis)

# ネットワーク構築
G = network_analysis.build_ppi_network(
    ppi_df=ppi_df,
    gwas_genes=gwas_genes,
    gene_scores=dict(zip(gene_scores_df['gene_symbol'], gene_scores_df['total_score'])),
)

# 中心性指標
centrality_df = network_analysis.compute_centrality_metrics(G)

# フロー伝播
flow_scores = network_analysis.propagate_flow_scores(
    G, gene_scores_df, damping=0.5, iterations=5
)

# RWR (バリアントスコアをシードとして使用)
rwr_scores = network_analysis.compute_rwr_scores(
    G, gene_scores_df, restart_prob=0.3, max_iter=100
)

# 統合スコア
integrated_scores_df = network_analysis.compute_integrated_scores(
    G, centrality_df, flow_scores, rwr_scores=rwr_scores
)

# 重要遺伝子選択
key_genes_df = network_analysis.select_key_genes(
    integrated_scores_df,
    top_n=TOP_N_KEY_GENES,
    min_score_percentile=60,
    include_all_gwas=True,
)
key_gene_list = key_genes_df['gene_symbol'].tolist()

print(f'\n最終重要遺伝子数: {len(key_gene_list)}')
display(key_genes_df.head(15))


---
## 6. エンリッチメント解析
重要遺伝子群を用いたエンリッチメント解析をローカルで実施:
- Reactome
- Gene Ontology (GO)
- HPO (Human Phenotype Ontology)

In [12]:
# 全 Enrichment 解析実行
enrichment_df = enrichment.run_all_enrichment(
    gene_list=key_gene_list,
    databases=ENRICHMENT_DBS,
    fdr_threshold=FDR_THRESHOLD,
)

if not enrichment_df.empty:
    # 結果保存
    enrichment_df.to_csv(
        os.path.join(config.OUTPUT_DIR, 'enrichment_results.csv'),
        index=False,
    )
    print(f'\n結果を保存: {config.OUTPUT_DIR}/enrichment_results.csv')
    enrichment_df.head(20)
else:
    print('有意なエンリッチメント結果なし')


Reactome Enrichment: 50 遺伝子
  → ローカル GMT 読み込み: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/data/ReactomePathways.gmt
  → Reactome パスウェイ: 2830 (ヒト)
[Reactome] 有意パスウェイ: 24

GO Enrichment: 50 遺伝子
  → ローカル GO GMT 読み込み: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/data/go_c5.gmt
  → GO ターム: 10454
[GO] 有意ターム: 132

HPO Enrichment: 50 遺伝子
  → ローカル HPO 読み込み: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/data/genes_to_phenotype.txt
  → HPO ターム: 10476
[HPO] 有意ターム: 121

[Enrichment] 全体結果: 277 有意ターム
  - HPO: 121 ターム
  - GO: 132 ターム
  - Reactome: 24 ターム

結果を保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_results.csv


In [13]:
# エンリッチメント可視化: データベースごとにバブルチャート
if not enrichment_df.empty:
    for db_name in enrichment_df['database'].unique():
        db_df = enrichment_df[enrichment_df['database'] == db_name].copy()
        if not db_df.empty:
            print(f'\n=== {db_name} Enrichment: {len(db_df)} 有意ターム ===')
            fig_bubble = visualization.create_enrichment_bubble_chart(
                db_df, top_n=20,
                save_path=os.path.join(config.OUTPUT_DIR, f'enrichment_bubble_{db_name.lower()}.html'),
            )
            fig_bubble.update_layout(title=f'{db_name} Enrichment (Top 20)')
            fig_bubble.show()


=== HPO Enrichment: 121 有意ターム ===
[Viz] バブルチャート保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_bubble_hpo.html



=== GO Enrichment: 132 有意ターム ===
[Viz] バブルチャート保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_bubble_go.html



=== Reactome Enrichment: 24 有意ターム ===
[Viz] バブルチャート保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_bubble_reactome.html


In [14]:
# エンリッチメント可視化: データベースごとに棒グラフ
if not enrichment_df.empty:
    for db_name in enrichment_df['database'].unique():
        db_df = enrichment_df[enrichment_df['database'] == db_name].copy()
        if not db_df.empty:
            fig_bar = visualization.create_enrichment_barplot(
                db_df, top_n=20,
                save_path=os.path.join(config.OUTPUT_DIR, f'enrichment_bar_{db_name.lower()}.html'),
            )
            fig_bar.update_layout(title=f'{db_name} Enrichment (-log10 FDR, Top 20)')
            fig_bar.show()

[Viz] 棒グラフ保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_bar_hpo.html


[Viz] 棒グラフ保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_bar_go.html


[Viz] 棒グラフ保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/enrichment_bar_reactome.html


---
## 7. 可視化
- サンキーダイアグラム: SNP → SNP関連遺伝子 → PPI重要遺伝子
- ネットワーク図 (Plotly)

In [15]:
# サンキーダイアグラム
fig_sankey = visualization.create_sankey_diagram(
    gwas_df=gwas_df,
    ppi_df=ppi_df,
    key_genes_df=key_genes_df,
    enrichment_df=enrichment_df if not enrichment_df.empty else None,
    top_n_pathways=15,
    save_path=os.path.join(config.OUTPUT_DIR, 'sankey_flow.html'),
)
fig_sankey.show()

[Viz] サンキーダイアグラム保存: /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output/sankey_flow.html


In [22]:
# PPI ネットワーク図 (Plotly)
fig_network = visualization.create_network_plot(
    G=G,
    key_genes=key_gene_list,
    gene_scores=flow_scores,
        known_targets=known_targets,
    save_path=os.path.join(config.OUTPUT_DIR, 'ppi_network_interactive.html'),
)
fig_network.show()

---
## 8. パスウェイ-遺伝子マトリックス
エンリッチメント解析の重要パスウェイごとに重要遺伝子をリストアップして示す。

In [24]:
# パスウェイ-遺伝子マトリックス
if not enrichment_df.empty:
    fig_matrix = visualization.create_pathway_gene_matrix(
        enrichment_df=enrichment_df,
        key_genes=key_gene_list,
    gene_scores=flow_scores,
        known_targets=known_targets,
        top_n_pathways=15,
        save_path=os.path.join(config.OUTPUT_DIR, 'pathway_gene_matrix.html'),
    )


In [25]:
# パスウェイごとの重要遺伝子リスト (テーブル)
if not enrichment_df.empty:
    print('\n=== パスウェイ別 重要遺伝子リスト ===')
    print('='*80)
    
    key_set = set(g.upper() for g in key_gene_list)
    
    for i, row in enrichment_df.head(20).iterrows():
        pw_genes = set(g.strip().upper() for g in row['genes'].split(','))
        key_in_pw = sorted(pw_genes & key_set)
        
        if key_in_pw:
            print(f'\n[{row["database"]}] {row["term_name"]}')
            print(f'  FDR: {row["fdr"]:.2e} | Fold: {row["fold_enrichment"]:.1f}x')
            print(f'  重要遺伝子 ({len(key_in_pw)}): {", ".join(key_in_pw)}')


=== パスウェイ別 重要遺伝子リスト ===

[HPO] Increased waist to hip ratio
  FDR: 1.37e-20 | Fold: 133.3x
  重要遺伝子 (12): GCK, HNF1A, HNF1B, HNF4A, IGF2BP2, IRS1, KCNJ11, MTNR1B, PPARG, SLC30A8, TCF7L2, WFS1

[HPO] Insulin resistance
  FDR: 1.36e-15 | Fold: 53.3x
  重要遺伝子 (12): GCK, HNF1A, HNF1B, HNF4A, IGF2BP2, IRS1, KCNJ11, MTNR1B, PPARG, SLC30A8, TCF7L2, WFS1

[HPO] Type II diabetes mellitus
  FDR: 9.54e-15 | Fold: 34.9x
  重要遺伝子 (13): GCK, HNF1A, HNF1B, HNF4A, IGF2BP2, IRS1, KCNJ11, MC4R, MTNR1B, PPARG, SLC30A8, TCF7L2, WFS1

[HPO] Late onset
  FDR: 3.55e-13 | Fold: 32.2x
  重要遺伝子 (12): GCK, HNF1A, HNF1B, HNF4A, IGF2BP2, IRS1, KCNJ11, MTNR1B, PPARG, SLC30A8, TCF7L2, WFS1

[GO] GOBP_INSULIN_SECRETION
  FDR: 5.91e-11 | Fold: 23.9x
  重要遺伝子 (12): ADCY5, GCK, HNF1A, HNF1B, HNF4A, IRS1, KCNJ11, LEP, MC4R, MTNR1B, SLC30A8, TCF7L2

[GO] GOBP_HORMONE_TRANSPORT
  FDR: 5.91e-11 | Fold: 17.2x
  重要遺伝子 (14): ADCY5, GCK, HNF1A, HNF1B, HNF4A, IRS1, KCNJ11, KCNQ1, LEP, MC4R, MTNR1B, PPARG, SLC30A8, TCF7L2

[GO] GOBP_

---
## 8.5 創薬ターゲット総合サマリーテーブル
重要遺伝子について、以下の情報を統合した一覧表を生成します:
- **統合スコア** (RWR + Flow + 中心性指標)
- **関連パスウェイ** (エンリッチメント解析から)
- **既知の創薬ターゲット情報** (ChEMBL / Pharos)
- **GWAS エビデンス** (SNP 由来かPPI由来か)

In [ ]:
# ========================================
# 創薬ターゲット総合サマリーテーブル
# ========================================

# 1. 重要遺伝子の統合スコアを取得
summary_df = integrated_scores_df.sort_values('integrated_score', ascending=False).head(500).copy()
summary_df = summary_df.reset_index(drop=True)

# 2. 各遺伝子が属するパスウェイを集約
if not enrichment_df.empty and 'genes' in enrichment_df.columns:
    gene_pathways = {}
    for _, row in enrichment_df.iterrows():
        pw_name = row.get('term_name', '')
        genes_in_pw = row.get('genes', [])
        if isinstance(genes_in_pw, str):
            genes_in_pw = [g.strip() for g in genes_in_pw.split(',')]
        for g in genes_in_pw:
            g_upper = g.upper()
            if g_upper not in gene_pathways:
                gene_pathways[g_upper] = []
            gene_pathways[g_upper].append(pw_name)
    summary_df['pathways'] = summary_df['gene_symbol'].apply(
        lambda g: ' | '.join(gene_pathways.get(g.upper(), ['-'])[:3])
    )
    summary_df['n_pathways'] = summary_df['gene_symbol'].apply(
        lambda g: len(gene_pathways.get(g.upper(), []))
    )
else:
    summary_df['pathways'] = '-'
    summary_df['n_pathways'] = 0

# 3. 既知ターゲット情報の付与
if known_targets:
    summary_df['drug_target'] = summary_df['gene_symbol'].apply(
        lambda g: known_targets.get(g.upper(), known_targets.get(g, ''))
    )
    summary_df['is_known_target'] = summary_df['drug_target'].apply(lambda x: '★' if x else '')
else:
    summary_df['drug_target'] = ''
    summary_df['is_known_target'] = ''

# 4. 表示用カラム選択
display_cols = [
    'gene_symbol', 'is_gwas', 'integrated_score',
    'rwr_score', 'flow_score', 'pagerank', 'betweenness_centrality', 'degree',
    'is_known_target', 'drug_target', 'n_pathways', 'pathways',
]
display_cols = [c for c in display_cols if c in summary_df.columns]
result_df = summary_df[display_cols].copy()

# カラム名を日本語に
rename_map = {
    'gene_symbol': '遺伝子',
    'is_gwas': 'GWAS',
    'integrated_score': '統合スコア',
    'rwr_score': 'RWRスコア',
    'flow_score': 'Flowスコア',
    'pagerank': 'PageRank',
    'betweenness_centrality': '媒介中心性',
    'degree': '次数',
    'is_known_target': '既知標的',
    'drug_target': '開発フェーズ',
    'n_pathways': 'PW数',
    'pathways': '関連パスウェイ (上位3)',
}
result_df = result_df.rename(columns=rename_map)

# 5. スタイル付き表示
print(f'\n=== 創薬ターゲット総合サマリー ({len(result_df)} 遺伝子) ===')
print(f'  うち既知ターゲット: {(result_df["既知標的"] == "★").sum()} 遺伝子')
print(f'  うちGWAS由来: {result_df["GWAS"].sum()} 遺伝子')
print()

styled = result_df.style \
    .background_gradient(subset=['統合スコア', 'RWRスコア'], cmap='YlOrRd') \
    .set_caption(f'創薬ターゲット総合サマリー: {DISEASE_TRAIT}') \
    .format({
        '統合スコア': '{:.4f}',
        'RWRスコア': '{:.6f}',
        'Flowスコア': '{:.2f}',
        'PageRank': '{:.4f}',
        '媒介中心性': '{:.4f}',
    })
display(styled)

# CSVとして保存
result_df.to_csv(os.path.join(config.OUTPUT_DIR, 'drug_target_summary.csv'), index=False)
print(f'\n保存: {config.OUTPUT_DIR}/drug_target_summary.csv')


---
## 9. データ保存

In [26]:
# 結果テーブルの保存
output_dir = config.OUTPUT_DIR

# GWAS SNP データ
gwas_df.to_csv(os.path.join(output_dir, 'gwas_snps.csv'), index=False)

# PPI インタラクション
if not ppi_df.empty:
    ppi_df.to_csv(os.path.join(output_dir, 'ppi_interactions.csv'), index=False)

# 遺伝子スコア
gene_scores_df.to_csv(os.path.join(output_dir, 'gene_scores.csv'), index=False)

# 統合スコア
if not integrated_scores_df.empty:
    integrated_scores_df.to_csv(os.path.join(output_dir, 'integrated_scores.csv'), index=False)

# 重要遺伝子
key_genes_df.to_csv(os.path.join(output_dir, 'key_genes.csv'), index=False)

print(f'✅ 全データを {output_dir} に保存完了')
print(f'\n保存ファイル一覧:')
for f in sorted(os.listdir(output_dir)):
    fpath = os.path.join(output_dir, f)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath)
        print(f'  {f} ({size:,} bytes)')

✅ 全データを /Users/yoshinorisatomi/Documents/antigravity/gwasppi/pj02/output に保存完了

保存ファイル一覧:
  enrichment_bar_go.html (4,848,148 bytes)
  enrichment_bar_hpo.html (4,848,008 bytes)
  enrichment_bar_reactome.html (4,848,385 bytes)
  enrichment_barplot.html (4,848,219 bytes)
  enrichment_bubble.html (4,851,324 bytes)
  enrichment_bubble_go.html (4,850,680 bytes)
  enrichment_bubble_hpo.html (4,850,378 bytes)
  enrichment_bubble_reactome.html (4,850,913 bytes)
  enrichment_results.csv (48,020 bytes)
  gene_scores.csv (24,320 bytes)
  gene_scores.html (4,851,274 bytes)
  gwas_snps.csv (6,665 bytes)
  integrated_scores.csv (110,068 bytes)
  key_genes.csv (13,389 bytes)
  pathway_gene_matrix.html (4,852,832 bytes)
  ppi_interactions.csv (31,664 bytes)
  ppi_network_interactive.html (4,959,398 bytes)
  sankey_flow.html (4,856,610 bytes)


## 10. パスウェイ別ネットワーク可視化
各エンリッチメントパスウェイに含まれる遺伝子だけを抽出し、PPIサブネットワークを描画します。

In [28]:
top_pw = enrichment_df.head(5)  # 上位5つのパスウェイを描画

for i, row in top_pw.iterrows():
    pw_name = row['term_name']
    pw_genes = [g.strip().upper() for g in row['genes'].split(',')]
    
    safe_name = "".join([c if c.isalnum() else "_" for c in pw_name[:30]])
    save_path = os.path.join(config.OUTPUT_DIR, f'pathway_net_{i+1}_{safe_name}.html')
    
    fig_pw = visualization.create_pathway_network_plot(
        G=G,
        pathway_name=pw_name,
        pathway_genes=pw_genes,
        gene_scores=flow_scores,
        known_targets=known_targets,
        save_path=save_path
    )
    if len(fig_pw.data) > 0:
        fig_pw.show()